# EKF Sensor Fusion Offline Analysis

This notebook reproduces the onboard Extended Kalman Filter (error-state, quaternion attitude)
by calling the **actual C functions** from `kalman_core.c` via `ctypes`.

**Workflow:**
1. Clone repo from GitHub into Colab
2. Compile `kalman_core.c` into a shared library with `make`
3. Load via ctypes and mirror the C structs in Python
4. Read time-series CSV data (IMU @ 200Hz, UWB @ 50Hz per anchor set)
5. Step through each measurement, calling predict/update exactly as the firmware does
6. Log and plot position, velocity, attitude, covariance over time

> You can edit C files directly in Colab and rerun the build cell to tune baked-in constants.


## 1. Setup: Clone Repository & Build Shared Library

Run the next cell to clone the firmware repository into Colab.

- If the repo is public, default settings work.
- If private, set `GITHUB_TOKEN` and use HTTPS token auth.
- Set `REPO_BRANCH` if you need a non-main branch.
- Place CSV files in `analysis/data/` inside the cloned repo (or upload there in Colab).


In [ ]:
import os
import shutil
import platform

# --- Git clone configuration ---
REPO_URL = 'https://github.com/aidanquandt/hybrid-localization-system.git'
REPO_BRANCH = 'eskf-analysis'
CLONE_PARENT = '/content'
CLONE_DIRNAME = 'hybrid-localization-system'
GITHUB_TOKEN = ''  # optional: set if repo is private

clone_target = os.path.join(CLONE_PARENT, CLONE_DIRNAME)
if os.path.exists(clone_target):
    shutil.rmtree(clone_target)

if GITHUB_TOKEN.strip():
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    auth_url = REPO_URL

cmd = f"git clone --depth 1 --branch {REPO_BRANCH} {auth_url} {clone_target}"
ret = os.system(cmd)
assert ret == 0, 'git clone failed. Check REPO_URL/REPO_BRANCH/token.'

REPO_ROOT = clone_target

# Locate analysis/ robustly (repo may be nested under one top-level folder)
def find_analysis_dir(root):
    direct = os.path.join(root, 'analysis')
    if os.path.isdir(direct):
        return direct

    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        # prune common heavy dirs for speed
        dirnames[:] = [d for d in dirnames if d not in {'.git', '.venv', '__pycache__', 'build', 'dist', 'node_modules'}]
        if os.path.basename(dirpath) == 'analysis':
            candidates.append(dirpath)

    if not candidates:
        return None

    # Prefer shallowest path under clone root
    candidates.sort(key=lambda p: p.count(os.sep))
    return candidates[0]

ANALYSIS_DIR = find_analysis_dir(REPO_ROOT)
assert ANALYSIS_DIR is not None, f'analysis/ not found anywhere under {REPO_ROOT}'

DATA_DIR = os.path.join(ANALYSIS_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Repo root:    {REPO_ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')
print(f'Data dir:     {DATA_DIR}')
print(f'Platform:     {platform.system()} {platform.machine()}')
print('Tip: edit C files in Colab and rerun build cell to apply baked-in parameter changes.')


In [ ]:
# Build the shared library from the cloned repo
os.chdir(ANALYSIS_DIR)
ret = os.system('make clean && make')
assert ret == 0, 'Build failed: check compiler output above.'

# Determine library path
if platform.system() == 'Darwin':
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.dylib')
else:
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.so')

assert os.path.exists(LIB_PATH), f'Library not found at {LIB_PATH}'
print(f'Library built: {LIB_PATH}')
print('If you change kalman_core.c or headers, rerun this cell before analysis.')


## 2. ctypes Bindings

In [ ]:
import ctypes
from ctypes import c_float, c_uint8, c_uint16, c_uint32, c_bool, POINTER, Structure, byref
import numpy as np

# --- Constants (must match kalman_core.h) ---
KC_STATE_DIM = 9

# State indices
KC_STATE_X  = 0  # Position X (world, m)
KC_STATE_Y  = 1  # Position Y (world, m)
KC_STATE_Z  = 2  # Position Z (world, m)
KC_STATE_PX = 3  # Velocity X (body, m/s)
KC_STATE_PY = 4  # Velocity Y (body, m/s)
KC_STATE_PZ = 5  # Velocity Z (body, m/s)
KC_STATE_D0 = 6  # Attitude error roll (rad)
KC_STATE_D1 = 7  # Attitude error pitch (rad)
KC_STATE_D2 = 8  # Attitude error yaw (rad)


# --- C struct mirrors ---

class ArmMatrixInstanceF32(Structure):
    """Mirrors arm_matrix_instance_f32"""
    _fields_ = [
        ('numRows', c_uint16),
        ('numCols', c_uint16),
        ('pData', POINTER(c_float)),
    ]


class Axis3f(Structure):
    """Mirrors Axis3f (3-axis float vector)"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
    ]


class DistanceMeasurement(Structure):
    """Mirrors distanceMeasurement_t"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
        ('distance', c_float),
        ('stdDev', c_float),
        ('anchorId', c_uint8),
    ]


class KalmanCoreParams(Structure):
    """Mirrors kalmanCoreParams_t"""
    _fields_ = [
        ('stdDevInitialPosition_xy', c_float),
        ('stdDevInitialPosition_z', c_float),
        ('stdDevInitialVelocity', c_float),
        ('stdDevInitialAttitude_rollpitch', c_float),
        ('stdDevInitialAttitude_yaw', c_float),
        ('procNoiseAcc_xy', c_float),
        ('procNoiseAcc_z', c_float),
        ('procNoiseVel', c_float),
        ('procNoisePos', c_float),
        ('procNoiseAtt', c_float),
        ('measNoiseGyro_rollpitch', c_float),
        ('measNoiseGyro_yaw', c_float),
        ('initialX', c_float),
        ('initialY', c_float),
        ('initialZ', c_float),
        ('initialYaw', c_float),
    ]


# Covariance matrix type: float[9][9]
CovMatrix = (c_float * KC_STATE_DIM) * KC_STATE_DIM
# Rotation matrix type: float[3][3]
RotMatrix = (c_float * 3) * 3


class KalmanCoreData(Structure):
    """Mirrors kalmanCoreData_t"""
    _fields_ = [
        ('S', c_float * KC_STATE_DIM),         # State vector
        ('q', c_float * 4),                     # Quaternion [w, x, y, z]
        ('R', RotMatrix),                       # Rotation matrix (body to world)
        ('P', CovMatrix),                       # Covariance matrix (9x9)
        ('Pm', ArmMatrixInstanceF32),           # ARM matrix instance for P
        ('initialQuaternion', c_float * 4),     # Initial quaternion
        ('isUpdated', c_bool),                  # Update flag
        ('lastPredictionMs', c_uint32),         # Last prediction timestamp
        ('lastProcessNoiseUpdateMs', c_uint32), # Last process noise timestamp
    ]


print(f'KalmanCoreData size: {ctypes.sizeof(KalmanCoreData)} bytes')
print(f'KalmanCoreParams size: {ctypes.sizeof(KalmanCoreParams)} bytes')

In [ ]:
# --- Load shared library and declare function signatures ---

lib = ctypes.CDLL(LIB_PATH)

# void kalmanCoreDefaultParams(kalmanCoreParams_t* params)
lib.kalmanCoreDefaultParams.argtypes = [POINTER(KalmanCoreParams)]
lib.kalmanCoreDefaultParams.restype = None

# void kalmanCoreInit(kalmanCoreData_t* kf, const kalmanCoreParams_t* params, uint32_t nowMs)
lib.kalmanCoreInit.argtypes = [POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32]
lib.kalmanCoreInit.restype = None

# void kalmanCorePredict(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                        Axis3f* acc, Axis3f* gyro, uint32_t nowMs)
lib.kalmanCorePredict.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams),
    POINTER(Axis3f), POINTER(Axis3f), c_uint32
]
lib.kalmanCorePredict.restype = None

# void kalmanCoreAddProcessNoise(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                                uint32_t nowMs)
lib.kalmanCoreAddProcessNoise.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32
]
lib.kalmanCoreAddProcessNoise.restype = None

# void kalmanCoreUpdateWithDistance(kalmanCoreData_t* kf, distanceMeasurement_t* d)
lib.kalmanCoreUpdateWithDistance.argtypes = [
    POINTER(KalmanCoreData), POINTER(DistanceMeasurement)
]
lib.kalmanCoreUpdateWithDistance.restype = None

# bool kalmanCoreFinalize(kalmanCoreData_t* kf)
lib.kalmanCoreFinalize.argtypes = [POINTER(KalmanCoreData)]
lib.kalmanCoreFinalize.restype = c_bool

# void kalmanCoreGetPosition(const kalmanCoreData_t* kf, float* x, float* y, float* z)
lib.kalmanCoreGetPosition.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetPosition.restype = None

# void kalmanCoreGetVelocity(const kalmanCoreData_t* kf, float* vx, float* vy, float* vz)
lib.kalmanCoreGetVelocity.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetVelocity.restype = None

# void kalmanCoreGetAttitude(const kalmanCoreData_t* kf, float* roll, float* pitch, float* yaw)
lib.kalmanCoreGetAttitude.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetAttitude.restype = None

# void kalmanCoreGetQuaternion(const kalmanCoreData_t* kf,
#                              float* qw, float* qx, float* qy, float* qz)
lib.kalmanCoreGetQuaternion.argtypes = [
    POINTER(KalmanCoreData),
    POINTER(c_float), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetQuaternion.restype = None

print('Library loaded and function signatures declared.')

## 3. Helper Functions

In [ ]:
def init_filter(params=None, init_timestamp_ms=0):
    """
    Initialize the Kalman filter with default or custom parameters.
    Returns (kf_data, kf_params) ctypes structs.
    """
    kf_params = KalmanCoreParams()
    lib.kalmanCoreDefaultParams(byref(kf_params))

    # Override with custom params if provided
    if params is not None:
        for field_name, _ in KalmanCoreParams._fields_:
            if field_name in params:
                setattr(kf_params, field_name, params[field_name])

    kf_data = KalmanCoreData()
    lib.kalmanCoreInit(byref(kf_data), byref(kf_params), c_uint32(int(init_timestamp_ms)))

    return kf_data, kf_params


def get_state(kf_data):
    """Extract full state from filter as a dict."""
    x, y, z = c_float(), c_float(), c_float()
    vx, vy, vz = c_float(), c_float(), c_float()
    roll, pitch, yaw = c_float(), c_float(), c_float()
    qw, qx, qy, qz = c_float(), c_float(), c_float(), c_float()

    lib.kalmanCoreGetPosition(byref(kf_data), byref(x), byref(y), byref(z))
    lib.kalmanCoreGetVelocity(byref(kf_data), byref(vx), byref(vy), byref(vz))
    lib.kalmanCoreGetAttitude(byref(kf_data), byref(roll), byref(pitch), byref(yaw))
    lib.kalmanCoreGetQuaternion(byref(kf_data), byref(qw), byref(qx), byref(qy), byref(qz))

    # Extract covariance diagonal
    P_diag = [kf_data.P[i][i] for i in range(KC_STATE_DIM)]

    return {
        'x': x.value, 'y': y.value, 'z': z.value,
        'vx': vx.value, 'vy': vy.value, 'vz': vz.value,
        'roll': roll.value, 'pitch': pitch.value, 'yaw': yaw.value,
        'qw': qw.value, 'qx': qx.value, 'qy': qy.value, 'qz': qz.value,
        'P_diag': P_diag,
    }


def process_imu(kf_data, kf_params, timestamp_ms, ax, ay, az, gx, gy, gz):
    """Process one IMU measurement: predict + process noise + finalize."""
    acc = Axis3f(ax, ay, az)
    gyro = Axis3f(gx, gy, gz)
    ts = c_uint32(int(timestamp_ms))

    lib.kalmanCorePredict(byref(kf_data), byref(kf_params), byref(acc), byref(gyro), ts)
    lib.kalmanCoreAddProcessNoise(byref(kf_data), byref(kf_params), ts)
    lib.kalmanCoreFinalize(byref(kf_data))


def process_uwb(kf_data, anchor_x, anchor_y, anchor_z, distance, stddev, anchor_id):
    """Process one UWB range measurement: update + finalize."""
    d = DistanceMeasurement(
        x=anchor_x, y=anchor_y, z=anchor_z,
        distance=distance, stdDev=stddev, anchorId=int(anchor_id)
    )
    lib.kalmanCoreUpdateWithDistance(byref(kf_data), byref(d))
    lib.kalmanCoreFinalize(byref(kf_data))


print('Helper functions defined.')


## 4. Load CSV Data

Current functional event format uses mixed event rows.

Required columns:
- `timestamp_ms`: timestamp in milliseconds.
- `type`: event type (`IMU`, `RANGING`, or `POSITION`).

IMU columns:
- `accel_x`, `accel_y`, `accel_z` (m/s^2)
- `gyro_x`, `gyro_y`, `gyro_z` (rad/s)

RANGING columns:
- `dist_m` (preferred) or legacy `distance`
- `anchor_addr` (preferred) or legacy `anchor_id`
- `anchor_x`, `anchor_y`, `anchor_z`
- optional `stddev` (defaults to `DEFAULT_UWB_STDDEV` if missing)

POSITION columns (reference only):
- `pos_x`, `pos_y`, `pos_z`
- `vel_x`, `vel_y`, `vel_z`
- `confidence`

Offline fusion behavior in this notebook:
- `IMU` rows run EKF prediction only when IMU is enabled (`imu_enable`, if logged).
- Every `RANGING` row runs EKF range update; when IMU is disabled, process noise is added on ranging timestamps (firmware-matching behavior).
- `POSITION` rows are not fused (reference/visualization + IMU enable timeline source).

Place your CSV file in `analysis/data/` and set the filename below.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# --- Set your CSV filename here ---
CSV_FILENAME = 'LOG008.CSV'  # <-- change filename here
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

if not os.path.exists(CSV_PATH):
    print(f'WARNING: CSV file not found at {CSV_PATH}')
    print('Place your data file in analysis/data/ and update CSV_FILENAME above.')
    print('Skipping data load - you can still use the helper functions manually.')
    df = None
else:
    df = pd.read_csv(CSV_PATH)
    if 'timestamp_ms' in df.columns:
        df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    print(f'Loaded {len(df)} rows from {CSV_FILENAME}')
    print(f'Columns: {list(df.columns)}')

    if len(df) > 0 and 'timestamp_ms' in df.columns:
        print(f'Time range: {df["timestamp_ms"].min()} - {df["timestamp_ms"].max()} ms')

    event_counts = df['type'].astype(str).str.upper().value_counts() if 'type' in df.columns else pd.Series(dtype=int)
    print(f'IMU samples: {int(event_counts.get("IMU", 0))}')
    print(f'RANGING samples: {int(event_counts.get("RANGING", 0))}')
    print(f'POSITION samples: {int(event_counts.get("POSITION", 0))}')
    print(f'Legacy UWB samples: {int(event_counts.get("UWB", 0))}')
    display(df.head(10))

    # Quick visualization of fused POSITION events directly from the CSV
    pos_df = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    if len(pos_df) > 0:
        pos_df = pos_df.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    # Ground truth rectangle: width=1.621 (x), height=1.835 (y), bottom-left shifted by (0.375, 1.553)
    gt_bl_x, gt_bl_y = 0.375, 1.553
    gt_w, gt_h = 1.621, 1.835
    gt_x = np.array([gt_bl_x, gt_bl_x + gt_w, gt_bl_x + gt_w, gt_bl_x, gt_bl_x])
    gt_y = np.array([gt_bl_y, gt_bl_y, gt_bl_y + gt_h, gt_bl_y + gt_h, gt_bl_y])

    # Anchor locations
    anchors = {
        'A4': (0.167, 4.237),
        'A3': (2.561, 4.254),
        'A6': (2.316, 0.000),
        'A8': (0.000, 0.000),
    }
    anchor_x = np.array([v[0] for v in anchors.values()])
    anchor_y = np.array([v[1] for v in anchors.values()])

    if len(pos_df) > 0:
        t_sec = (pos_df['timestamp_ms'].to_numpy() - pos_df['timestamp_ms'].iloc[0]) / 1000.0
        x = pos_df['pos_x'].to_numpy()
        y = pos_df['pos_y'].to_numpy()

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))

        if len(pos_df) > 1:
            points = np.array([x, y]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            norm = Normalize(vmin=t_sec.min(), vmax=t_sec.max())
            lc = LineCollection(segments, cmap='turbo', norm=norm)
            lc.set_array(t_sec[:-1])
            lc.set_linewidth(2.0)
            line = ax.add_collection(lc)
            cbar = fig.colorbar(line, ax=ax)
            cbar.set_label('Time since first plotted POSITION sample (s)')
            ax.plot(x[0], y[0], 'go', markersize=8, label='Start')
            ax.plot(x[-1], y[-1], 'rs', markersize=8, label='End')
        else:
            ax.plot(x[0], y[0], 'bo', markersize=8, label='Only POSITION sample')

        # Overlay ground truth path
        ax.plot(gt_x, gt_y, 'k--', linewidth=2.0, label='Ground truth path')

        # Overlay anchors
        ax.scatter(anchor_x, anchor_y, marker='^', s=70, c='magenta', edgecolors='black', label='Anchors')
        for name, (ax_x, ax_y) in anchors.items():
            ax.text(ax_x + 0.03, ax_y + 0.03, name, fontsize=9, color='black')

        ax.set_xlabel('pos_x (m)')
        ax.set_ylabel('pos_y (m)')
        ax.set_title('CSV POSITION 2D Trajectory + Ground Truth + Anchors (full run)')
        ax.grid(True, alpha=0.3)

        # Make axes square with equal x/y scale and square frame, covering path + GT + anchors
        all_x = np.concatenate([x, gt_x, anchor_x])
        all_y = np.concatenate([y, gt_y, anchor_y])
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        cx, cy = 0.5 * (x_min + x_max), 0.5 * (y_min + y_max)
        span = max(x_max - x_min, y_max - y_min)
        if span == 0:
            span = 1.0
        pad = 0.08 * span
        half = 0.5 * span + pad
        ax.set_xlim(-1.0, 13.0)
        ax.set_ylim(-1.0, 6.0)
        ax.set_aspect('equal', adjustable='box')

        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()
    else:
        print('No POSITION rows with valid timestamp_ms/pos_x/pos_y.')


## 5. Streamlined Static Analysis

This section runs a compact offline analysis with static plots only:
1. Colored onboard POSITION history (full run)
2. Offline EKF with UWB-only
3. Offline EKF with UWB+IMU
4. 2D comparisons (offline vs offline, and offline vs onboard)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

DEFAULT_UWB_STDDEV = 0.25
X_LIM = (-5.0, 25.0)
Y_LIM = (-5.0, 15.0)


def _get_numeric(row, candidates):
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return float(row[col])
            except (TypeError, ValueError):
                pass
    return None


def _get_int(row, candidates):
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return int(float(row[col]))
            except (TypeError, ValueError):
                pass
    return None


def _to_boolish(v, default=False):
    if pd.isna(v):
        return default
    s = str(v).strip().lower()
    if s in {'1', 'true', 't', 'yes', 'y', 'on'}:
        return True
    if s in {'0', 'false', 'f', 'no', 'n', 'off'}:
        return False
    return default


def _build_imu_enable_lookup(df):
    required = {'type', 'timestamp_ms', 'imu_enable'}
    if not required.issubset(df.columns):
        return None, None

    pos = df[df['type'].astype(str).str.upper() == 'POSITION'][['timestamp_ms', 'imu_enable']].copy()
    if len(pos) == 0:
        return None, None

    pos['timestamp_ms'] = pd.to_numeric(pos['timestamp_ms'], errors='coerce')
    pos = pos.dropna(subset=['timestamp_ms']).sort_values('timestamp_ms', kind='stable')
    if len(pos) == 0:
        return None, None

    vals = pos['imu_enable'].apply(lambda x: _to_boolish(x, default=False)).to_numpy(dtype=bool)
    ts = pos['timestamp_ms'].to_numpy(dtype=np.int64)
    keep = np.r_[ts[1:] != ts[:-1], True]
    return ts[keep], vals[keep]


def run_filter_mode(df, use_imu=True, custom_params=None, log_interval=1, respect_imu_enable=False):
    if df is None or len(df) == 0:
        return pd.DataFrame()

    ordered_df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    # IMPORTANT: process the full raw event history (no POSITION-based clipping).
    ordered_df = ordered_df.dropna(subset=['timestamp_ms']).reset_index(drop=True)
    if len(ordered_df) == 0:
        return pd.DataFrame()

    init_ts = int(float(ordered_df['timestamp_ms'].iloc[0]))
    kf_data, kf_params = init_filter(custom_params, init_timestamp_ms=init_ts)

    # Use IMU timeline only if explicitly encoded in data; default to enabled.
    imu_ts, imu_vals = _build_imu_enable_lookup(ordered_df) if respect_imu_enable else (None, None)
    has_imu_timeline = imu_ts is not None and len(imu_ts) > 0

    log = {'timestamp_ms': [], 'type': [], 'x': [], 'y': [], 'z': []}

    fused_count = 0
    for _, row in ordered_df.iterrows():
        ts = int(float(row['timestamp_ms']))
        mtype = str(row['type']).strip().upper()

        imu_enabled_current = True
        if has_imu_timeline:
            j = np.searchsorted(imu_ts, ts, side='right') - 1
            # Before first POSITION record, keep IMU enabled so full raw IMU history is used.
            imu_enabled_current = bool(imu_vals[j]) if j >= 0 else True

        did_fuse = False

        if mtype in ('RANGING', 'UWB'):
            dist = _get_numeric(row, ['dist_m', 'distance'])
            anchor_x = _get_numeric(row, ['anchor_x'])
            anchor_y = _get_numeric(row, ['anchor_y'])
            anchor_z = _get_numeric(row, ['anchor_z'])
            anchor_id = _get_int(row, ['anchor_addr', 'anchor_id'])
            stddev = _get_numeric(row, ['stddev'])
            if stddev is None:
                stddev = DEFAULT_UWB_STDDEV

            if None not in (dist, anchor_x, anchor_y, anchor_z, anchor_id):
                # UWB-only mode requested: do update-only (no process-noise injection).
                if use_imu and (not imu_enabled_current):
                    lib.kalmanCoreAddProcessNoise(byref(kf_data), byref(kf_params), c_uint32(ts))
                process_uwb(kf_data, anchor_x, anchor_y, anchor_z, dist, stddev, anchor_id)
                did_fuse = True

        elif use_imu and mtype == 'IMU':
            if not imu_enabled_current:
                continue
            ax = _get_numeric(row, ['accel_x'])
            ay = _get_numeric(row, ['accel_y'])
            az = _get_numeric(row, ['accel_z'])
            gx = _get_numeric(row, ['gyro_x'])
            gy = _get_numeric(row, ['gyro_y'])
            gz = _get_numeric(row, ['gyro_z'])
            if None not in (ax, ay, az, gx, gy, gz):
                process_imu(kf_data, kf_params, ts, ax, ay, az, gx, gy, gz)
                did_fuse = True

        if did_fuse:
            fused_count += 1
            if fused_count % log_interval == 0:
                state = get_state(kf_data)
                log['timestamp_ms'].append(ts)
                log['type'].append(mtype)
                log['x'].append(state['x'])
                log['y'].append(state['y'])
                log['z'].append(state['z'])

    out = pd.DataFrame(log)
    print(f"mode={'UWB+IMU' if use_imu else 'UWB-only'}: logged {len(out)} states from {fused_count} fused events over {len(ordered_df)} input rows")
    if len(out) > 0:
        t0 = int(out['timestamp_ms'].iloc[0]); t1 = int(out['timestamp_ms'].iloc[-1])
        print(f"output horizon: {t0} -> {t1} ms")
    return out


In [ ]:
# 1) Full POSITION history from CSV, color-coded by time
if df is None:
    print('No data loaded.')
else:
    pos_df = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_df = pos_df.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_df) < 2:
        print('Need at least 2 POSITION samples.')
    else:
        t = (pos_df['timestamp_ms'].to_numpy(dtype=float) - float(pos_df['timestamp_ms'].iloc[0])) / 1000.0
        x = pos_df['pos_x'].to_numpy(dtype=float)
        y = pos_df['pos_y'].to_numpy(dtype=float)

        pts = np.array([x, y]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)

        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        norm = Normalize(vmin=float(np.min(t[:-1])), vmax=float(np.max(t[:-1])))
        lc = LineCollection(segs, cmap='turbo', norm=norm, linewidths=2.0)
        lc.set_array(t[:-1])
        ax.add_collection(lc)
        cbar = fig.colorbar(lc, ax=ax)
        cbar.set_label('Time (s)')

        ax.plot(x[0], y[0], 'go', markersize=8, label='Start')
        ax.plot(x[-1], y[-1], 'rs', markersize=8, label='End')
        ax.set_title('Onboard POSITION History (full run)')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_xlim(*X_LIM)
        ax.set_ylim(*Y_LIM)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()


In [ ]:
# 2) Run offline EKF twice: UWB-only and UWB+IMU, then compare in 2D
if df is None:
    print('No data loaded.')
else:
    results_uwb_only = run_filter_mode(df, use_imu=False)
    results_uwb_imu = run_filter_mode(df, use_imu=True)

    if len(results_uwb_only) < 2 or len(results_uwb_imu) < 2:
        print(f'Insufficient output: uwb_only={len(results_uwb_only)}, uwb_imu={len(results_uwb_imu)}')
    else:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.plot(results_uwb_only['x'], results_uwb_only['y'], color='tab:orange', linewidth=1.5, label='Offline UWB-only')
        ax.plot(results_uwb_imu['x'], results_uwb_imu['y'], color='tab:blue', linewidth=1.5, label='Offline UWB+IMU')
        ax.set_title('Offline EKF Comparison: UWB-only vs UWB+IMU')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_xlim(*X_LIM)
        ax.set_ylim(*Y_LIM)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()


In [ ]:
# 3) Compare offline UWB+IMU vs onboard POSITION in 2D
if df is None:
    print('No data loaded.')
elif 'results_uwb_imu' not in globals() or results_uwb_imu is None or len(results_uwb_imu) < 2:
    print('Run the previous cell first to generate `results_uwb_imu`.')
else:
    pos_csv = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_csv = pos_csv.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_csv) < 2:
        print('Need at least 2 onboard POSITION samples for comparison.')
    else:
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        ax.plot(pos_csv['pos_x'].to_numpy(dtype=float), pos_csv['pos_y'].to_numpy(dtype=float),
                color='tab:green', linewidth=1.5, label='Onboard POSITION (MCU)')
        ax.plot(results_uwb_imu['x'].to_numpy(dtype=float), results_uwb_imu['y'].to_numpy(dtype=float),
                color='tab:blue', linewidth=1.5, label='Offline UWB+IMU')
        ax.set_title('Onboard vs Offline Sensor Fusion (UWB+IMU)')
        ax.set_xlabel('x (m)')
        ax.set_ylabel('y (m)')
        ax.set_xlim(*X_LIM)
        ax.set_ylim(*Y_LIM)
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3)
        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()
